# Does (slope, R²) of the recent trend predict the forward move?

Single asset, single horizon, no zigzag. For each minute snapshot we fit a line to
the recent `look_back` of **normalized vwap**, extract its **slope** and **R²**, then
look `look_ahead` minutes ahead and ask whether price moved in the slope's direction.

The study follows two project specifications:

- **`idea_look_back_look_ahead.md`** — windowing & snapshot model. Here we use the
  simple `look_back + look_ahead` window (not 1440+240). Load boundaries are extended
  in **exclusive** mode so every in-range snapshot has a full look-back behind and a
  full look-ahead ahead.
- **`idea_normalize_based_on_last_price_clip.md`** — the `k`/clip normalization, with
  `price_l` anchored to the **last look-back candle**.

Outputs:
1. A **21 × 11** directional-accuracy heatmap (slope rows, R² columns), plus companion
   grids for sample count `N` and continuation magnitude.
2. **Confusion matrices over an R² gate**, split by slope sign, swept over
   `τ ∈ {0.0 … 0.9}`, with precision / recall / accuracy / take-rate tables and
   precision/recall-vs-τ curves; plus a per-R²-decile continued / reversed / timed-out
   breakdown.

In [ ]:
%pip install numpy pandas pyarrow requests huggingface_hub numba plotly

In [ ]:
import os
import sys

# 1. Define your GitHub repository details
REPO_URL    = "https://github.com/payamdav/pycrypto.git"
REPO_NAME   = "pycrypto"
BRANCH_NAME = "claude/pensive-einstein-ui4c7e"  # this notebook lives on a non-main branch

# 2. Clone the specific branch if it hasn't been cloned yet
if not os.path.exists(REPO_NAME):
    !git clone -b {BRANCH_NAME} {REPO_URL}

# 3. Add the cloned repository root to the Python path
REPO_PATH = os.path.abspath(REPO_NAME)
if REPO_PATH not in sys.path:
    sys.path.append(REPO_PATH)

In [ ]:
import io
import re
import math
import numpy as np
import pandas as pd
import requests
import pyarrow.parquet as pq
import numba as nb
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta, timezone
from huggingface_hub import HfApi

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

## Configuration

The 7 available tickers (see `agents/datasets/assets.md`): `btcusdt`, `ethusdt`,
`trumpusdt`, `vineusdt`, `adausdt`, `xrpusdt`, `dogeusdt`. Exactly one is uncommented.

In [ ]:
# --- pick exactly ONE asset (uncomment one) ---
ASSET = "btcusdt"
# ASSET = "ethusdt"
# ASSET = "trumpusdt"
# ASSET = "vineusdt"
# ASSET = "adausdt"
# ASSET = "xrpusdt"
# ASSET = "dogeusdt"

look_back     = 180      # candles used for the slope/R2 regression
look_ahead    = 240      # candles used for the forward vwap
target_return = 0.0020   # 20 bps; barrier threshold on the forward return
k             = 100      # normalization scale: +/-0.01 (1%) -> +/-1 after clip
clip          = 1.0      # hard clip bound for normalized prices: [-clip, +clip]
stride        = 1        # snapshot step in minutes (see "Overlap" note)
time_frame    = 1        # minutes per candle

# study date range (inclusive). Per-snapshot window = look_back + look_ahead candles.
START = "2026-03-01"
END   = "2026-03-31"

# slope normalization divisor (derived): 2 * 1440 / look_back
SLOPE_DIVISOR = 2 * 1440.0 / look_back
print(f"ASSET={ASSET}  look_back={look_back}  look_ahead={look_ahead}")
print(f"SLOPE_DIVISOR = 2*1440/look_back = {SLOPE_DIVISOR}  (=> slope_raw is divided by this)")

## Data loading (extended boundaries, exclusive mode)

We load `vwap`, `q`, `v` and `ts`. Per `idea_look_back_look_ahead.md` exclusive mode,
extend the load window by `look_back` candles behind `START` and `look_ahead` candles
ahead of `END`, so every snapshot whose anchor `ts ∈ [START, END]` has a full
look-back and full look-ahead.

In [ ]:
REPO = "https://huggingface.co/datasets/payamdavaee/candles/resolve/main"

def load_range(asset: str, start: str, end: str, columns=None) -> pd.DataFrame:
    """Load candle rows for [start, end] inclusive (UTC) from the HF candles dataset."""
    api = HfApi()
    files = api.list_repo_files("payamdavaee/candles", repo_type="dataset")
    pattern = re.compile(rf"^{asset}/{asset}-1m-(\d{{4}})-(\d{{2}})\.parquet$")

    start_ts = int(datetime.fromisoformat(start).replace(tzinfo=timezone.utc).timestamp() * 1000)
    end_ts   = int(datetime.fromisoformat(end).replace(tzinfo=timezone.utc).timestamp() * 1000) + 86_400_000

    frames = []
    for f in sorted(files):
        m = pattern.match(f)
        if not m:
            continue
        url  = f"{REPO}/{f}"
        resp = requests.get(url, timeout=120)
        resp.raise_for_status()
        tbl  = pq.read_table(io.BytesIO(resp.content), columns=columns or None)
        df   = tbl.to_pandas()
        ts   = df["ts"].astype("int64")
        df   = df[(ts >= start_ts) & (ts < end_ts)]
        if not df.empty:
            frames.append(df)

    if not frames:
        return pd.DataFrame()
    return pd.concat(frames).sort_values("ts").reset_index(drop=True)


tf_min = time_frame
dt_start = datetime.fromisoformat(START).replace(tzinfo=timezone.utc)
dt_end   = datetime.fromisoformat(END).replace(tzinfo=timezone.utc)

# exclusive-mode boundary extension (indicator_extra = 0 here)
load_start = dt_start - timedelta(minutes=look_back  * tf_min)
load_end   = dt_end   + timedelta(minutes=look_ahead * tf_min)

print("Loading", ASSET,
      "from", load_start.strftime("%Y-%m-%d"),
      "to",   load_end.strftime("%Y-%m-%d"))

df = load_range(ASSET,
                load_start.strftime("%Y-%m-%d"),
                load_end.strftime("%Y-%m-%d"),
                columns=["ts", "vwap", "q", "v"])
df = df.sort_values("ts").reset_index(drop=True)
print("loaded rows:", len(df))
df.head()

## Per-snapshot computation

For each snapshot anchored at the **last look-back candle**:

1. **look-back → slope and R²** — OLS on the normalized/clipped vwap series
   `y = clip(k·(vwap/current_price − 1), −clip, clip)` against
   `x = arange(look_back) / 1440`. Slope is then normalized to `[−1, 1]` via
   `slope = clip(slope_raw / (2·1440/look_back), −1, 1)`.
2. **look-ahead → forward vwap** — `forward_vwap = Σq / Σv` over the next
   `look_ahead` candles; `raw_fwd_return = forward_vwap / current_price − 1`;
   `forward_vwap_norm = clip(k·raw_fwd_return, −clip, clip)`.
3. **barrier label** from the raw return vs `±target_return`:
   `+1` top / `−1` bottom / `0` time-out.

The core loop is JITed with numba; everything vectorizes cleanly over snapshots.

In [ ]:
@nb.njit(cache=True)
def compute_snapshots(vwap, q, v, look_back, look_ahead, k, clip, slope_divisor,
                      target_return, stride):
    """Vectorized-over-snapshots OLS slope/R2 + forward vwap + barrier label.

    Anchor index a runs over [look_back-1, n-look_ahead-1] in steps of `stride`.
    Returns parallel arrays for each output column.
    """
    n = vwap.shape[0]
    first_a = look_back - 1
    last_a  = n - look_ahead - 1          # inclusive
    if last_a < first_a:
        m = 0
    else:
        m = (last_a - first_a) // stride + 1

    idx_anchor   = np.empty(m, dtype=np.int64)
    cur_price    = np.empty(m, dtype=np.float64)
    fwd_vwap     = np.empty(m, dtype=np.float64)
    raw_ret      = np.empty(m, dtype=np.float64)
    fwd_norm     = np.empty(m, dtype=np.float64)
    slope_out    = np.empty(m, dtype=np.float64)
    r2_out       = np.empty(m, dtype=np.float64)
    barrier_out  = np.empty(m, dtype=np.int64)

    # fixed x axis (time in 1440-min units) and its statistics
    x = np.empty(look_back, dtype=np.float64)
    for i in range(look_back):
        x[i] = i / 1440.0
    xmean = 0.0
    for i in range(look_back):
        xmean += x[i]
    xmean /= look_back
    sxx = 0.0
    for i in range(look_back):
        dx = x[i] - xmean
        sxx += dx * dx

    out_i = 0
    a = first_a
    while a <= last_a:
        current_price = vwap[a]
        cur_price[out_i] = current_price
        idx_anchor[out_i] = a

        # --- look-back: build normalized/clipped y, OLS slope + R2 ---
        ymean = 0.0
        for j in range(look_back):
            yv = k * (vwap[a - look_back + 1 + j] / current_price - 1.0)
            if yv > clip:
                yv = clip
            elif yv < -clip:
                yv = -clip
            ymean += yv
        ymean /= look_back

        sxy = 0.0
        syy = 0.0
        for j in range(look_back):
            yv = k * (vwap[a - look_back + 1 + j] / current_price - 1.0)
            if yv > clip:
                yv = clip
            elif yv < -clip:
                yv = -clip
            dx = x[j] - xmean
            dy = yv - ymean
            sxy += dx * dy
            syy += dy * dy

        if sxx > 0.0:
            slope_raw = sxy / sxx
        else:
            slope_raw = 0.0
        if syy > 0.0:
            r2 = (sxy * sxy) / (sxx * syy)
        else:
            r2 = 0.0
        if r2 < 0.0:
            r2 = 0.0
        elif r2 > 1.0:
            r2 = 1.0

        s = slope_raw / slope_divisor
        if s > 1.0:
            s = 1.0
        elif s < -1.0:
            s = -1.0
        slope_out[out_i] = s
        r2_out[out_i] = r2

        # --- look-ahead: forward vwap = sum(q)/sum(v) over next look_ahead ---
        sumq = 0.0
        sumv = 0.0
        for j in range(look_ahead):
            sumq += q[a + 1 + j]
            sumv += v[a + 1 + j]
        if sumv > 0.0:
            fv = sumq / sumv
        else:
            fv = current_price
        fwd_vwap[out_i] = fv

        rr = fv / current_price - 1.0
        raw_ret[out_i] = rr
        fn = k * rr
        if fn > clip:
            fn = clip
        elif fn < -clip:
            fn = -clip
        fwd_norm[out_i] = fn

        if rr >= target_return:
            barrier_out[out_i] = 1
        elif rr <= -target_return:
            barrier_out[out_i] = -1
        else:
            barrier_out[out_i] = 0

        out_i += 1
        a += stride

    return (idx_anchor, cur_price, fwd_vwap, raw_ret, fwd_norm,
            slope_out, r2_out, barrier_out)

In [ ]:
vwap_arr = np.ascontiguousarray(df["vwap"].to_numpy(dtype=np.float64))
q_arr    = np.ascontiguousarray(df["q"].to_numpy(dtype=np.float64))
v_arr    = np.ascontiguousarray(df["v"].to_numpy(dtype=np.float64))
ts_arr   = df["ts"].to_numpy(dtype=np.int64)

(idx_anchor, cur_price, fwd_vwap, raw_ret, fwd_norm,
 slope_out, r2_out, barrier_out) = compute_snapshots(
    vwap_arr, q_arr, v_arr, look_back, look_ahead,
    float(k), float(clip), float(SLOPE_DIVISOR), float(target_return), int(stride))

print("snapshots computed:", len(idx_anchor))

### Sample table

One row per snapshot. We keep only snapshots whose **anchor timestamp** falls inside
the requested `[START, END]` study range (the extra leading/trailing candles existed
only to guarantee complete windows).

In [ ]:
anchor_ts = ts_arr[idx_anchor]

tbl = pd.DataFrame({
    "timestamp"        : pd.to_datetime(anchor_ts, unit="ms", utc=True),
    "ts"               : anchor_ts,
    "current_price"    : cur_price,
    "forward_vwap_240" : fwd_vwap,
    "raw_fwd_return"   : raw_ret,
    "forward_vwap_norm": fwd_norm,
    "slope"            : slope_out,
    "r2"               : r2_out,
    "barrier"          : barrier_out,
})

start_ms = int(dt_start.timestamp() * 1000)
end_ms   = int(dt_end.timestamp() * 1000) + 86_400_000   # inclusive end-day
in_range = (tbl["ts"] >= start_ms) & (tbl["ts"] < end_ms)
tbl = tbl.loc[in_range].reset_index(drop=True)

print("in-range snapshots:", len(tbl))
tbl.head()

In [ ]:
# save the full per-snapshot table
out_base = f"snapshots_{ASSET}_lb{look_back}_la{look_ahead}"
try:
    tbl.to_parquet(out_base + ".parquet", index=False)
    print("saved", out_base + ".parquet")
except Exception as e:
    print("parquet save skipped:", e)
tbl.to_csv(out_base + ".csv", index=False)
print("saved", out_base + ".csv")

In [ ]:
# quick sanity / distribution summary
print(tbl[["slope", "r2", "raw_fwd_return", "forward_vwap_norm"]].describe())
print("\nbarrier counts:")
print(tbl["barrier"].value_counts().sort_index())

## Output 1 — the 21 × 11 directional-accuracy table

Grid: **slope on rows** (−1.0 → 1.0, step 0.1 → 21 bins), **R² on columns**
(0.0 → 1.0, step 0.1 → 11 bins). Bin by rounding `slope` and `r2` to one decimal
(after the clip). Each cell holds the **percentage of samples in that bin whose forward
direction matches the slope direction**:

```
match = sign(raw_fwd_return) == sign(slope)
cell  = 100 * mean(match)   over samples in the bin
```

We also render two companion grids: (a) the per-cell sample count `N` (sparse cells
< 50 are masked), and (b) the mean `raw_fwd_return × sign(slope)` (continuation
magnitude). The **zero-slope row** (`slope = 0.0`) has undefined direction; we report
it as the fraction with `raw_fwd_return > 0` and flag it separately.

In [ ]:
SLOPE_BINS = np.round(np.arange(-1.0, 1.0 + 1e-9, 0.1), 1)   # 21 rows
R2_BINS    = np.round(np.arange(0.0, 1.0 + 1e-9, 0.1), 1)        # 11 cols
N_ROW, N_COL = len(SLOPE_BINS), len(R2_BINS)
MIN_N = 50   # mask cells below this many samples

slope_b = np.round(tbl["slope"].to_numpy(), 1)
slope_b = np.clip(slope_b, -1.0, 1.0)
r2_b    = np.round(tbl["r2"].to_numpy(), 1)
r2_b    = np.clip(r2_b, 0.0, 1.0)

slope_idx = np.rint((slope_b + 1.0) / 0.1).astype(int)   # 0..20
r2_idx    = np.rint(r2_b / 0.1).astype(int)               # 0..10
slope_idx = np.clip(slope_idx, 0, N_ROW - 1)
r2_idx    = np.clip(r2_idx, 0, N_COL - 1)

sgn_slope = np.sign(tbl["slope"].to_numpy())
sgn_ret   = np.sign(tbl["raw_fwd_return"].to_numpy())
match     = (sgn_ret == sgn_slope).astype(float)
cont_mag  = tbl["raw_fwd_return"].to_numpy() * sgn_slope   # continuation magnitude

acc_grid  = np.full((N_ROW, N_COL), np.nan)
n_grid    = np.zeros((N_ROW, N_COL))
mag_grid  = np.full((N_ROW, N_COL), np.nan)

zero_row = np.where(np.isclose(SLOPE_BINS, 0.0))[0][0]

for ri in range(N_ROW):
    for ci in range(N_COL):
        cell = (slope_idx == ri) & (r2_idx == ci)
        nc = int(cell.sum())
        n_grid[ri, ci] = nc
        if nc == 0:
            continue
        if ri == zero_row:
            # zero slope: direction undefined -> fraction with raw_fwd_return > 0
            acc_grid[ri, ci] = 100.0 * float((tbl["raw_fwd_return"].to_numpy()[cell] > 0).mean())
        else:
            acc_grid[ri, ci] = 100.0 * float(match[cell].mean())
        mag_grid[ri, ci] = float(cont_mag[cell].mean())

print("filled cells:", int((n_grid > 0).sum()), "/", N_ROW * N_COL)
print(f"zero-slope row (slope=0.0) reports P(raw_fwd_return>0); interpret separately.")

In [ ]:
# mask sparse cells for the displayed accuracy/magnitude grids
acc_disp = np.where(n_grid >= MIN_N, acc_grid, np.nan)
mag_disp = np.where(n_grid >= MIN_N, mag_grid, np.nan)

xlabels = [f"{c:.1f}" for c in R2_BINS]
ylabels = [f"{s:.1f}" for s in SLOPE_BINS]

fig1 = go.Figure(data=go.Heatmap(
    z=acc_disp, x=xlabels, y=ylabels,
    colorscale="RdBu", zmid=50.0, zmin=0, zmax=100,
    colorbar=dict(title="% match"),
    hovertemplate="slope=%{y}<br>R2=%{x}<br>dir-acc=%{z:.1f}%<extra></extra>",
))
fig1.update_layout(
    title=f"Output 1 — Directional accuracy % ({ASSET}, lb={look_back}, la={look_ahead}, target={target_return})<br>"
          f"<sub>cells with N < {MIN_N} masked; slope=0.0 row = P(raw_fwd_return>0)</sub>",
    xaxis_title="R2 bin", yaxis_title="slope bin",
    width=720, height=820,
)
fig1.show()

In [ ]:
# companion grid (a): sample count N per cell (sparse cells highlighted via mask)
n_disp = np.where(n_grid >= MIN_N, n_grid, np.nan)
fig1b = go.Figure(data=go.Heatmap(
    z=n_disp, x=xlabels, y=ylabels,
    colorscale="Viridis",
    colorbar=dict(title="N"),
    hovertemplate="slope=%{y}<br>R2=%{x}<br>N=%{z}<extra></extra>",
))
fig1b.update_layout(
    title=f"Output 1 companion (a) — sample count N per (slope, R2) cell<br>"
          f"<sub>cells with N < {MIN_N} masked (shown blank)</sub>",
    xaxis_title="R2 bin", yaxis_title="slope bin",
    width=720, height=820,
)
fig1b.show()

In [ ]:
# companion grid (b): mean continuation magnitude raw_fwd_return * sign(slope)
mmax = np.nanmax(np.abs(mag_disp)) if np.isfinite(np.nanmax(np.abs(mag_disp))) else 0.01
fig1c = go.Figure(data=go.Heatmap(
    z=mag_disp, x=xlabels, y=ylabels,
    colorscale="RdBu", zmid=0.0, zmin=-mmax, zmax=mmax,
    colorbar=dict(title="mean cont. return"),
    hovertemplate="slope=%{y}<br>R2=%{x}<br>mean cont=%{z:.5f}<extra></extra>",
))
fig1c.update_layout(
    title=f"Output 1 companion (b) — mean continuation magnitude (raw_fwd_return × sign(slope))<br>"
          f"<sub>cells with N < {MIN_N} masked</sub>",
    xaxis_title="R2 bin", yaxis_title="slope bin",
    width=720, height=820,
)
fig1c.show()

## Output 2 — confusion matrices over an R² gate

Split by slope sign, with the "trend validated" truth:

- **Positive slope:** truth `T = (raw_fwd_return >= +target_return)`
- **Negative slope:** truth `T = (raw_fwd_return <= -target_return)`

For each slope sign sweep an R² threshold `τ ∈ {0.0, 0.1, …, 0.9}`. The decision is
"trust the trend if it's clean enough": `D = (r2 >= τ)`.

```
TP = #(D=1, T=1)   FP = #(D=1, T=0)
FN = #(D=0, T=1)   TN = #(D=0, T=0)
```

This yields 10 confusion matrices per slope sign (20 total). For each we tabulate
precision, recall, accuracy and take-rate, then plot precision & recall vs τ.

In [ ]:
TAUS = np.round(np.arange(0.0, 0.9 + 1e-9, 0.1), 1)   # 10 values

def gate_table(sub, truth):
    """sub: subset DataFrame (one slope sign); truth: boolean array aligned to sub."""
    r2v = sub["r2"].to_numpy()
    N = len(sub)
    rows = []
    cms  = {}
    for tau in TAUS:
        D = r2v >= tau
        TP = int(np.sum(D & truth))
        FP = int(np.sum(D & ~truth))
        FN = int(np.sum(~D & truth))
        TN = int(np.sum(~D & ~truth))
        prec = TP / (TP + FP) if (TP + FP) > 0 else np.nan
        rec  = TP / (TP + FN) if (TP + FN) > 0 else np.nan
        acc  = (TP + TN) / N if N > 0 else np.nan
        take = (TP + FP) / N if N > 0 else np.nan
        rows.append({"tau": tau, "TP": TP, "FP": FP, "FN": FN, "TN": TN,
                     "precision": prec, "recall": rec, "accuracy": acc, "take_rate": take})
        cms[tau] = np.array([[TP, FP], [FN, TN]])
    return pd.DataFrame(rows), cms

pos = tbl[tbl["slope"] > 0].copy()
neg = tbl[tbl["slope"] < 0].copy()

truth_pos = (pos["raw_fwd_return"].to_numpy() >= target_return)
truth_neg = (neg["raw_fwd_return"].to_numpy() <= -target_return)

pos_tab, pos_cms = gate_table(pos, truth_pos)
neg_tab, neg_cms = gate_table(neg, truth_neg)

print(f"Positive-slope subset N = {len(pos)} (truth base-rate = {truth_pos.mean():.3f})")
display(pos_tab.style.format({"precision": "{:.3f}", "recall": "{:.3f}",
                              "accuracy": "{:.3f}", "take_rate": "{:.3f}"}))

In [ ]:
print(f"Negative-slope subset N = {len(neg)} (truth base-rate = {truth_neg.mean():.3f})")
display(neg_tab.style.format({"precision": "{:.3f}", "recall": "{:.3f}",
                              "accuracy": "{:.3f}", "take_rate": "{:.3f}"}))

In [ ]:
# Render all 20 confusion matrices as annotated heatmaps (2 rows x 10 cols per sign).
def cm_grid(cms, title):
    fig = make_subplots(rows=1, cols=len(TAUS),
                        subplot_titles=[f"τ={t:.1f}" for t in TAUS],
                        horizontal_spacing=0.012)
    for i, tau in enumerate(TAUS):
        cm = cms[tau]
        fig.add_trace(go.Heatmap(
            z=cm[::-1], x=["T=1", "T=0"], y=["D=0", "D=1"],
            colorscale="Blues", showscale=False,
            text=cm[::-1], texttemplate="%{text}", textfont=dict(size=9),
            hovertemplate="%{y},%{x}: %{z}<extra></extra>"),
            row=1, col=i + 1)
    fig.update_layout(title=title, height=320, width=1500)
    return fig

cm_grid(pos_cms, f"Output 2 — confusion matrices, POSITIVE slope ({ASSET})").show()
cm_grid(neg_cms, f"Output 2 — confusion matrices, NEGATIVE slope ({ASSET})").show()

In [ ]:
# precision & recall vs tau, one curve per slope sign
fig2 = make_subplots(rows=1, cols=2, subplot_titles=("Precision vs τ", "Recall vs τ"))

fig2.add_trace(go.Scatter(x=pos_tab["tau"], y=pos_tab["precision"], mode="lines+markers",
                          name="pos slope", line=dict(color="royalblue")), row=1, col=1)
fig2.add_trace(go.Scatter(x=neg_tab["tau"], y=neg_tab["precision"], mode="lines+markers",
                          name="neg slope", line=dict(color="firebrick")), row=1, col=1)
# baseline precision (tau=0 take-everything continuation rate)
fig2.add_hline(y=float(pos_tab["precision"].iloc[0]), line_dash="dot", line_color="royalblue",
               row=1, col=1)
fig2.add_hline(y=float(neg_tab["precision"].iloc[0]), line_dash="dot", line_color="firebrick",
               row=1, col=1)

fig2.add_trace(go.Scatter(x=pos_tab["tau"], y=pos_tab["recall"], mode="lines+markers",
                          name="pos slope", line=dict(color="royalblue"), showlegend=False), row=1, col=2)
fig2.add_trace(go.Scatter(x=neg_tab["tau"], y=neg_tab["recall"], mode="lines+markers",
                          name="neg slope", line=dict(color="firebrick"), showlegend=False), row=1, col=2)

fig2.update_xaxes(title_text="R2 threshold τ", row=1, col=1)
fig2.update_xaxes(title_text="R2 threshold τ", row=1, col=2)
fig2.update_yaxes(title_text="precision", row=1, col=1)
fig2.update_yaxes(title_text="recall", row=1, col=2)
fig2.update_layout(title=f"Output 2 — precision & recall vs τ ({ASSET}); dotted = τ=0 baseline",
                   width=1000, height=480)
fig2.show()

### Per-R²-decile breakdown (continued / reversed / timed-out)

A per-bin (rather than cumulative-gate) view: for each R² decile and slope sign, the
share of outcomes that **continued** (barrier in the slope's direction), **reversed**
(barrier against the slope), or **timed-out** (barrier 0).

In [ ]:
def decile_breakdown(sub, sign):
    """sign = +1 (positive slope) or -1 (negative slope)."""
    r2d = np.clip(np.rint(sub["r2"].to_numpy() / 0.1).astype(int), 0, 10)  # 0..10 -> deciles
    bar = sub["barrier"].to_numpy()
    rows = []
    for d in range(11):
        cell = r2d == d
        n = int(cell.sum())
        if n == 0:
            rows.append({"r2_bin": round(d * 0.1, 1), "N": 0,
                         "continued_%": np.nan, "reversed_%": np.nan, "timed_out_%": np.nan})
            continue
        b = bar[cell]
        cont = float(np.mean(b == sign)) * 100
        rev  = float(np.mean(b == -sign)) * 100
        timo = float(np.mean(b == 0)) * 100
        rows.append({"r2_bin": round(d * 0.1, 1), "N": n,
                     "continued_%": cont, "reversed_%": rev, "timed_out_%": timo})
    return pd.DataFrame(rows)

pos_dec = decile_breakdown(pos, +1)
neg_dec = decile_breakdown(neg, -1)

print("POSITIVE slope — outcome shares per R2 decile:")
display(pos_dec.style.format({"continued_%": "{:.1f}", "reversed_%": "{:.1f}", "timed_out_%": "{:.1f}"}))
print("NEGATIVE slope — outcome shares per R2 decile:")
display(neg_dec.style.format({"continued_%": "{:.1f}", "reversed_%": "{:.1f}", "timed_out_%": "{:.1f}"}))

In [ ]:
# stacked outcome-share bars per R2 decile, side by side for both slope signs
fig3 = make_subplots(rows=1, cols=2, subplot_titles=("Positive slope", "Negative slope"))
for col, dec in [(1, pos_dec), (2, neg_dec)]:
    x = dec["r2_bin"]
    fig3.add_trace(go.Bar(x=x, y=dec["continued_%"], name="continued",
                          marker_color="seagreen", showlegend=(col == 1)), row=1, col=col)
    fig3.add_trace(go.Bar(x=x, y=dec["reversed_%"], name="reversed",
                          marker_color="firebrick", showlegend=(col == 1)), row=1, col=col)
    fig3.add_trace(go.Bar(x=x, y=dec["timed_out_%"], name="timed-out",
                          marker_color="lightgray", showlegend=(col == 1)), row=1, col=col)
    fig3.update_xaxes(title_text="R2 decile", row=1, col=col)
fig3.update_yaxes(title_text="% of outcomes", row=1, col=1)
fig3.update_layout(barmode="stack", title=f"Per-R2-decile outcome shares ({ASSET})",
                   width=1000, height=480)
fig3.show()

## How to read the result (interpretation guide)

- **Output 1 flat ~50% everywhere** → slope & R² carry little directional information
  at this horizon for this asset. **Brightens toward high R² and/or steep slope** →
  they do; the bright region is where the trend tends to continue. Watch for
  **non-monotonicity** — accuracy may *peak then fall* at the steepest/cleanest extremes
  (exhaustion), in which case the loyal region is interior, not at the corner.
- **Output 2:** if precision rises with `τ`, the R² gate is real — cleaner trends
  continue more often. If precision is flat in `τ`, R² adds nothing and only slope sign
  matters. Compare both directions; they need not be symmetric. `τ = 0.0` is the
  take-everything baseline (precision = unconditional continuation rate, shown dotted).

### Notes

- **Overlap:** at `stride = 1`, consecutive snapshots share almost all of their look-back
  and forward windows, so the descriptive percentages are reliable but the *effective*
  sample count is far below the row count — don't attach naive significance to small
  differences. For honest error bars, re-run with `stride ≥ look_ahead` (non-overlapping
  forward windows). For these tables, small `stride` is fine.
- **Single asset, single horizon by design.** `look_back`, `look_ahead`, `target_return`,
  `k`, and `ASSET` are the only knobs; everything else is derived.